In [ ]:
!which python

In [ ]:
!pip install cmake==3.27.0 pyarrow==16.1.0 datasets transformers accelerate sagemaker boto3 --upgrade --no-cache-dir

In [ ]:
!pip install peft

In [ ]:
!pip install -U bitsandbytes

In [ ]:
import boto3

In [ ]:
s3=boto3.client("s3")

In [ ]:
response = s3.list_objects_v2(Bucket="llm-finetune-dataset-suman", Prefix="datasets/")

In [ ]:
for obj in response.get("Contents", []):
    print(obj)

In [ ]:
for obj in response.get("Contents", []):
    print(obj["Key"])

In [ ]:
dataset_path = "s3://llm-finetune-dataset-suman/datasets/pharma_instruction_data.csv/"

In [ ]:
from datasets import load_dataset

In [ ]:
dataset = load_dataset("csv",data_files={"train":dataset_path},split="train")
# dataset = load_dataset("csv", data_files="/content/pharma_instruction_data.csv",split="train")

In [ ]:
dataset

In [ ]:
print(dataset)

In [ ]:
print(dataset[0])

In [ ]:
def format_example(example):
    prompt = f"### Instruction:\n{example['instruction']}\n### Input:\n{example['input']}\n### Response:\n{example['output']}"
    return {"text": prompt}

In [ ]:
dataset = dataset.map(format_example)

In [ ]:
dataset

In [ ]:
dataset['text'][0]

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType

In [ ]:

model_id = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model)

In [ ]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def tokenize_fn(example):
    tokens = tokenizer(example["text"], truncation=True, padding="max_length", max_length=512)
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

In [ ]:
tokenized = dataset.map(tokenize_fn, batched=True)

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

In [ ]:
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none"
)

In [ ]:
model = get_peft_model(model, lora_config)

In [ ]:
args = TrainingArguments(
    output_dir="./tinyllama-instruction",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_total_limit=1,
    report_to="none"

In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized,

In [ ]:

trainer.train()

In [ ]:

model_path = "/content/tinyllama-instruction/checkpoint-3"

In [ ]:

instruction_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

In [ ]:
prompt = "Explain the mechanism of action of Metformin."

In [ ]:
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

In [ ]:
outputs = instruction_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

In [ ]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))